In [ ]:
from models.captura_tela import CapturaTela
from models.automacao_cliques import AutomacaoOCR
import time

# 1) Focar janela Import
auto_ocr = AutomacaoOCR('Import')  # já foca no __init__
captura = auto_ocr.captura

if not captura.janela_atual:
    raise SystemExit("Janela 'Import' não encontrada. Abra o ONESOURCE na tela da fatura e rode de novo.")


In [ ]:
# 2) COORDENADAS INICIAIS CHUTADAS DA GRADE (vamos calibrar)
#    olhando sua imagem, a grade começa logo abaixo do "Part Number" do filtro.
#    Ajuste fino depois de olhar o PNG.
X_GRADE = 240   # começa um pouco à direita da borda interna
Y_GRADE = 510   # logo abaixo dos campos Núm. Ordem / Part Number do filtro
LARGURA_GRADE = 900
ALTURA_GRADE  = 180

print(f"\n📸 Capturando região da grade: ({X_GRADE}, {Y_GRADE}, {LARGURA_GRADE}, {ALTURA_GRADE})")

img = captura.capturar_regiao(
    X_GRADE, Y_GRADE,
    LARGURA_GRADE, ALTURA_GRADE,
    salvar=True,
    nome_arquivo='debug_grade_itens.png'
)

if not img:
    raise SystemExit("Falha ao capturar região da grade.")

print("💾 Imagem da grade salva como 'debug_grade_itens.png'.")

In [ ]:
# 3) (OPCIONAL) Rodar OCR só pra ver o que ele enxerga nessa região
print("\n🔍 Rodando OCR na região da grade...")

resultado_ocr = auto_ocr.processar_ocr(
    regiao=(X_GRADE, Y_GRADE, LARGURA_GRADE, ALTURA_GRADE),
    forcar_nova=True
)

if not resultado_ocr:
    raise SystemExit("Falha no OCR da grade.")

data = resultado_ocr['data']

print("\n📜 Textos detectados na grade:")
for i in range(len(data['text'])):
    texto = data['text'][i].strip()
    try:
        conf = int(data['conf'][i])
    except ValueError:
        conf = 0

    if not texto:
        continue

    print(f"  '{texto}' (conf={conf}%)")

print("\n✅ Debug de grade concluído. Abra 'debug_grade_itens.png' e ajuste X_GRADE/Y_GRADE/LARGURA_GRADE/ALTURA_GRADE conforme necessário.")
